# Module 6 · Demo — MCP Servers

**From 0 to Agentic AI — DataHack Summit 2026**

So far every tool was a Python function *inside* our app. **MCP** (Model Context Protocol) is a
**standard way to connect tools from separate servers** — use ready-made ones, or expose your own.
One protocol, many servers; add a data source without rewriting the agent.

### What you'll do
1. **Expose your own** MCP server — a tiny company-ops server (`stdio`)
2. **Connect** to it with `langchain-mcp-adapters` → its functions become LangChain tools
3. Hand those tools to an agent — same `create_agent` as always

---
## Setup

In [1]:
# Install the workshop stack (Colab). Locally, use `uv sync` instead.
# Version ranges match src/pyproject.toml.
!pip install -q "langchain>=1.2,<2" "langchain-openai>=1.1,<2" "langgraph>=1.0,<2" \
               "langchain-mcp-adapters>=0.1" "mcp>=1.0"

ERROR: Could not find a version that satisfies the requirement langchain<2,>=1.2 (from versions: 0.0.1, 0.0.2, 0.0.3, 0.0.4, 0.0.5, 0.0.6, 0.0.7, 0.0.8, 0.0.9, 0.0.10, 0.0.11, 0.0.12, 0.0.13, 0.0.14, 0.0.15, 0.0.16, 0.0.17, 0.0.18, 0.0.19, 0.0.20, 0.0.21, 0.0.22, 0.0.23, 0.0.24, 0.0.25, 0.0.26, 0.0.27, 0.0.28, 0.0.29, 0.0.30, 0.0.31, 0.0.32, 0.0.33, 0.0.34, 0.0.35, 0.0.36, 0.0.37, 0.0.38, 0.0.39, 0.0.40, 0.0.41, 0.0.42, 0.0.43, 0.0.44, 0.0.45, 0.0.46, 0.0.47, 0.0.48, 0.0.49, 0.0.50, 0.0.51, 0.0.52, 0.0.53, 0.0.54, 0.0.55, 0.0.56, 0.0.57, 0.0.58, 0.0.59, 0.0.60, 0.0.61, 0.0.63, 0.0.64, 0.0.65, 0.0.66, 0.0.67, 0.0.68, 0.0.69, 0.0.70, 0.0.71, 0.0.72, 0.0.73, 0.0.74, 0.0.75, 0.0.76, 0.0.77, 0.0.78, 0.0.79, 0.0.80, 0.0.81, 0.0.82, 0.0.83, 0.0.84, 0.0.85, 0.0.86, 0.0.87, 0.0.88, 0.0.89, 0.0.90, 0.0.91, 0.0.92, 0.0.93, 0.0.94, 0.0.95, 0.0.96, 0.0.97, 0.0.98, 0.0.99rc0, 0.0.99, 0.0.100, 0.0.101rc0, 0.0.101, 0.0.102rc0, 0.0.102, 0.0.103, 0.0.104, 0.0.105, 0.0.106, 0.0.107, 0.0.108, 0.0.109, 0.0

In [2]:
import os
from getpass import getpass

try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))
except Exception:
    pass

for key in ["OPENAI_API_KEY"]:
    if not os.environ.get(key):
        os.environ[key] = getpass(f"{key}: ")

---
## Step 1 · Expose your own MCP server

An MCP server is just a script that registers tools. `FastMCP` makes it a few lines. We write it
to a file so it can run as a separate process (this cell uses `%%writefile`).

In [3]:
%%writefile company_ops_server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("company-ops")

@mcp.tool()
def get_headcount(department: str) -> str:
    """Return the headcount for a company department."""
    data = {"billing": 12, "platform": 30, "data": 8}
    return f"{department}: {data.get(department.lower(), 'unknown')} people"

@mcp.tool()
def get_office(city: str) -> str:
    """Return the office address for a city."""
    offices = {"bengaluru": "The Leela, Bengaluru", "milan": "Via Roma 1, Milan"}
    return offices.get(city.lower(), "no office there")

if __name__ == "__main__":
    mcp.run(transport="stdio")

Overwriting company_ops_server.py


---
## Step 2 · Connect to the server

`MultiServerMCPClient` launches the server and turns its tools into LangChain tools. It's async,
so we `await`. (You can connect to **many** servers here — hence *Multi*.)

In [4]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient({
    "company-ops": {
        "command": "python",
        "args": ["company_ops_server.py"],
        "transport": "stdio",
    }
})

mcp_tools = await client.get_tools()
print("tools from server:", [t.name for t in mcp_tools])

tools from server: ['get_headcount', 'get_office']


### Call one directly
These behave like any other LangChain tool (async `.ainvoke`).

In [5]:
res = await mcp_tools[0].ainvoke({"department": "platform"})
print(res)

[{'type': 'text', 'text': 'platform: 30 people', 'id': 'lc_5d7d9a99-0341-4cea-8d34-926adc6b1819'}]


---
## Step 3 · Give the MCP tools to an agent

No special handling — MCP tools are just tools. The agent uses them like the local ones.

In [6]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent

llm = init_chat_model("gpt-4.1-mini", model_provider="openai", temperature=0)
agent = create_agent(llm, tools=mcp_tools)

out = await agent.ainvoke({"messages": [("user", "How many people are on the platform team?")]})
print(out["messages"][-1].content)

There are 30 people on the platform team.


---
## Key takeaways
- **MCP** is a standard socket: connect tools from separate servers instead of hand-wiring each.
- **Expose your own** with `FastMCP`, or point the client at **existing** servers — same client.
- To the agent, MCP tools are **just tools** — modular, swappable, scalable architecture.

➡️ **Next (Module 7):** when one agent juggling many tools isn't enough — **multi-agent systems**.